In [1]:
from maap.maap import MAAP
maap = MAAP(maap_host="api.maap-project.org")

import boto3
import os

In [5]:
results = maap.searchGranule(cmr_host="cmr.earthdata.nasa.gov",
                                short_name='SENTINEL-1A_DP_GRD_HIGH',
                                #bounding_box='-124.8136026553671,32.445063449213436,-113.75989347462286,42.24498423828791',
                                producer_granule_id="S1A_IW_GRDH_1SDV_20250330T171421_20250330T171446_058537_073E4F_985B",
                                limit=10,
                                #temporal='2023-06-01T00:00:00Z,2030-06-12T23:59:59Z'
                                )

In [6]:
results

[{'concept-id': 'G3471862300-ASF',
  'collection-concept-id': 'C1214470533-ASF',
  'revision-id': '3',
  'format': 'application/echo10+xml',
  'Granule': {'{http://www.w3.org/2001/XMLSchema-instance}noNamespaceSchemaLocation': '',
   'GranuleUR': 'S1A_IW_GRDH_1SDV_20250330T171421_20250330T171446_058537_073E4F_985B-GRD_HD',
   'InsertTime': '2025-03-30T19:05:57Z',
   'LastUpdate': '2025-03-30T19:05:57Z',
   'Collection': {'ShortName': 'SENTINEL-1A_DP_GRD_HIGH', 'VersionId': '1'},
   'DataGranule': {'SizeMBDataGranule': '845.4925422668457',
    'ProducerGranuleId': 'S1A_IW_GRDH_1SDV_20250330T171421_20250330T171446_058537_073E4F_985B',
    'DayNightFlag': 'UNSPECIFIED',
    'ProductionDateTime': '2025-03-30T17:14:21.899737Z'},
   'PGEVersionClass': {'PGEName': 'Sentinel-1 IPF', 'PGEVersion': '003.91'},
   'Temporal': {'RangeDateTime': {'BeginningDateTime': '2025-03-30T17:14:21.899737Z',
     'EndingDateTime': '2025-03-30T17:14:46.898759Z'}},
   'Spatial': {'HorizontalSpatialDomain': {'Geo

In [7]:
test = results[0].getDownloadUrl()
test

's3://asf-ngap2w-p-s1-grd-7d1b4348/S1A_IW_GRDH_1SDV_20250330T171421_20250330T171446_058537_073E4F_985B.zip'

In [8]:
os.path.basename(test)

'S1A_IW_GRDH_1SDV_20250330T171421_20250330T171446_058537_073E4F_985B.zip'

In [9]:
os.path.dirname(test).replace('s3://','')

'asf-ngap2w-p-s1-grd-7d1b4348'

In [10]:
split_path = os.path.split(test)
bucket = split_path[0].replace('s3://','')
prefix = '/'.join(split_path[1:])

In [11]:
prefix

'S1A_IW_GRDH_1SDV_20250330T171421_20250330T171446_058537_073E4F_985B.zip'

In [12]:
bucket

'asf-ngap2w-p-s1-grd-7d1b4348'

In [14]:
def get_s3_creds(url):
    return maap.aws.earthdata_s3_credentials(url)

def get_s3_client(s3_cred_endpoint):
    creds=get_s3_creds(s3_cred_endpoint)
    boto3_session = boto3.Session(
            aws_access_key_id=creds['accessKeyId'],
            aws_secret_access_key=creds['secretAccessKey'],
            aws_session_token=creds['sessionToken']
    )
    return boto3_session.client("s3")

def download_s3_file(s3, bucket, file_name):
    os.makedirs("/projects/local_data/", exist_ok=True) # create directories, as necessary
    download_path=f"/projects/local_data/{file_name}"
    s3.download_file(bucket, f"{file_name}", download_path)
    return download_path

In [15]:
asf_s3 = "https://sentinel1.asf.alaska.edu/s3credentials"
creds = get_s3_creds(asf_s3)


In [16]:
s3 = get_s3_client(asf_s3)
#bucket = 'asf-ngap2w-p-s1-slc-7b420b89'
download_path = download_s3_file(s3, bucket, os.path.basename(test))
download_path

'/projects/local_data/S1A_IW_GRDH_1SDV_20250330T171421_20250330T171446_058537_073E4F_985B.zip'